In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime, timedelta

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# =====================================================================
# BƯỚC 1: ĐỌC VÀ GỘP 6 FILE CSV SẢN PHẨM THỜI TRANG TIKI
# =====================================================================
print("--- BƯỚC 1: ĐANG GỘP CÁC FILE CSV SẢN PHẨM ---")
csv_files = glob.glob(os.path.join(RAW_DIR, "vietnamese_tiki_products_*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(f"Không tìm thấy file CSV nào bắt đầu bằng 'vietnamese_tiki_products_' trong thư mục {RAW_DIR}!")

df_list = []
for file_path in csv_files:
    df = pd.read_csv(file_path)
    # Trích xuất phân loại lớn dựa trên tên file (ví dụ: backpacks_suitcases, fashion_accessories,...)
    category_name = os.path.basename(file_path).replace("vietnamese_tiki_products_", "").replace(".csv", "")
    df['sub_category'] = category_name
    df_list.append(df)
    print(f"-> Đã đọc file: {os.path.basename(file_path)} ({len(df)} sản phẩm)")


products_df = pd.concat(df_list, ignore_index=True)

# Xử lý làm sạch nhẹ dữ liệu sản phẩm gốc để làm trọng số sinh log
products_df = products_df.dropna(subset=['id'])
products_df['id'] = products_df['id'].astype(int)
products_df = products_df.drop_duplicates(subset=['id'])
if "Unnamed: 0" in products_df.columns:
    products_df = products_df.drop(columns=["Unnamed: 0"])
products_df["quantity_sold"] = products_df["quantity_sold"].fillna(0)
products_df["review_count"] = products_df["review_count"].fillna(0)
products_df["rating_average"] = products_df["rating_average"].fillna(0)
products_df["popularity_weight"] = (
    0.7 * np.log1p(products_df["quantity_sold"]) +
    0.2 * np.log1p(products_df["review_count"]) +
    0.1 * (products_df["rating_average"] / 5) +
    1
)
# Xuất ra thư mục processed để làm Metadata phục vụ Mapping hiển thị kết quả sau này
products_df.to_csv(os.path.join(PROCESSED_DIR, "products.csv"), index=False)
print(f" Thống kê: Tổng cộng gộp được {len(products_df)} sản phẩm độc nhất. Đã lưu vào data/processed/products.csv\n")


# =====================================================================
# BƯỚC 2: SINH DỮ LIỆU LOG HÀNH VI NGƯỜI DÙNG HỢP LOGIC (MOCK INTERACTIONS)
# =====================================================================
print("--- BƯỚC 2: ĐANG SINH LOG HÀNH VI NGƯỜI DÙNG (VIEW - CART - PURCHASE) ---")
np.random.seed(42)  # Đảm bảo kết quả cố định mỗi lần chạy

NUM_USERS = 3000   # Thiết lập quy mô số lượng User giả lập cho hệ thống
user_ids = [f"USR_{i:04d}" for i in range(1, NUM_USERS + 1)]

# Phân nhóm danh sách Item ID theo từng danh mục để phục vụ việc gán sở thích (Persona)
categories = products_df['sub_category'].unique()
items_by_cat = {cat: products_df[products_df['sub_category'] == cat]['id'].values for cat in categories}

# Tính toán mức độ phổ biến toàn cục của sản phẩm dựa trên số lượng đánh giá và điểm trung bình
# Sản phẩm hot hơn sẽ có xác suất được tương tác (Click/Mua) cao hơn
products_df['popularity_weight'] = products_df['review_count'] * (products_df['rating_average'] / 5.0) + 1.0

# Chuẩn hóa trọng số xác suất cho từng nhóm sản phẩm
weights_by_cat = {}
for cat in categories:
    sub_df = products_df[products_df['sub_category'] == cat]
    cat_weights = sub_df['popularity_weight'].values
    weights_by_cat[cat] = cat_weights / cat_weights.sum()

# Chỉ định ngẫu nhiên một "Sở thích chủ đạo" (Persona) cho mỗi User
user_personas = np.random.choice(categories, size=NUM_USERS)
user_persona_dict = dict(zip(user_ids, user_personas))

interaction_logs = []
start_date = datetime(2026, 6, 1) # Mốc thời gian giả lập

for user_id in user_ids:
    primary_cat = user_persona_dict[user_id]
    
    # Số lượng sản phẩm khác nhau mà user này sẽ tương tác (Ví dụ từ 5 đến 30 sản phẩm)
    num_distinct_items = np.random.randint(5, 31)
    
    selected_items = []
    for _ in range(num_distinct_items):
        # LOGIC 1: Quy luật sở thích nhóm (80% tương tác trúng danh mục yêu thích, 20% đi xem dạo danh mục khác)
        if np.random.rand() < 0.8:
            target_cat = primary_cat
        else:
            target_cat = np.random.choice(categories)
            
        # LOGIC 2: Trong cùng danh mục, ưu tiên chọn sản phẩm có lượt review/rating cao hơn (hàng bán chạy)
        chosen_item = np.random.choice(items_by_cat[target_cat], p=weights_by_cat[target_cat])
        selected_items.append(chosen_item)
    
    # Loại bỏ trùng lặp sản phẩm trong cùng một phiên hành vi của user này
    selected_items = list(set(selected_items))
    
    # LOGIC 3: Phễu chuyển đổi hành vi thực tế (View -> Cart -> Purchase)
    for item_id in selected_items:
        # Thời gian ngẫu nhiên trong tháng diễn ra hành vi tương tác
        timestamp = start_date + timedelta(
            days=np.random.randint(0, 30), 
            hours=np.random.randint(0, 24), 
            minutes=np.random.randint(0, 60)
        )
        
        # Mọi sản phẩm được chú ý chắc chắn phải có hành vi 'view' (Xem từ 1 đến 4 lần)
        num_views = np.random.randint(1, 5)
        for _ in range(num_views):
            interaction_logs.append({
                'user_id': user_id,
                'item_id': item_id,
                'event_type': 'view',
                'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S')
            })
            timestamp += timedelta(minutes=np.random.randint(1, 15)) # Cách nhau vài phút hành vi
            
        # Tỷ lệ thêm vào giỏ hàng ('cart') - Giả định trung bình khoảng 25% các mặt hàng đã xem
        is_carted = np.random.rand() < 0.25
        if is_carted:
            interaction_logs.append({
                'user_id': user_id,
                'item_id': item_id,
                'event_type': 'cart',
                'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S')
            })
            timestamp += timedelta(minutes=np.random.randint(5, 45))
            
            # Tỷ lệ mua hàng ('purchase') - Chỉ xảy ra khi đã bỏ vào giỏ (Chiếm khoảng 40% tỷ lệ chuyển đổi từ giỏ hàng)
            is_purchased = np.random.rand() < 0.40
            if is_purchased:
                interaction_logs.append({
                    'user_id': user_id,
                    'item_id': item_id,
                    'event_type': 'purchase',
                    'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S')
                })
        else:
            # Trường hợp mua ngay không qua giỏ hàng (Tỷ lệ cực kỳ thấp ~2%)
            if np.random.rand() < 0.02:
                interaction_logs.append({
                    'user_id': user_id,
                    'item_id': item_id,
                    'event_type': 'purchase',
                    'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S')
                })

# Tạo DataFrame từ list kết quả
interactions_df = pd.DataFrame(interaction_logs)

# Trộn ngẫu nhiên thứ tự các dòng dữ liệu để log trông tự nhiên theo chuỗi thời gian đan xen của các user
interactions_df = interactions_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Xuất ra file csv đích
interactions_df.to_csv(os.path.join(PROCESSED_DIR, "mock_interactions.csv"), index=False)

print(f" Sinh dữ liệu thành công!")
print(f"-> Tổng số bản ghi log tương tác sinh ra: {len(interactions_df)} dòng")
print(f"-> Tỷ lệ phân phối hành vi:\n{interactions_df['event_type'].value_counts(normalize=True)}")
print(f"Đã lưu file log sạch vào: data/processed/mock_interactions.csv")

--- BƯỚC 1: ĐANG GỘP CÁC FILE CSV SẢN PHẨM ---
-> Đã đọc file: vietnamese_tiki_products_backpacks_suitcases.csv (5361 sản phẩm)
-> Đã đọc file: vietnamese_tiki_products_fashion_accessories.csv (16019 sản phẩm)
-> Đã đọc file: vietnamese_tiki_products_men_bags.csv (4234 sản phẩm)
-> Đã đọc file: vietnamese_tiki_products_men_shoes.csv (5745 sản phẩm)
-> Đã đọc file: vietnamese_tiki_products_women_bags.csv (4325 sản phẩm)
-> Đã đọc file: vietnamese_tiki_products_women_shoes.csv (5919 sản phẩm)
 Thống kê: Tổng cộng gộp được 41576 sản phẩm độc nhất. Đã lưu vào data/processed/products.csv

--- BƯỚC 2: ĐANG SINH LOG HÀNH VI NGƯỜI DÙNG (VIEW - CART - PURCHASE) ---
 Sinh dữ liệu thành công!
-> Tổng số bản ghi log tương tác sinh ra: 146287 dòng
-> Tỷ lệ phân phối hành vi:
event_type
view        0.872326
cart        0.087417
purchase    0.040256
Name: proportion, dtype: float64
Đã lưu file log sạch vào: data/processed/mock_interactions.csv
